In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Mga default na setting at mga opsyon sa configuration ng transpilation


<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na kinakailangan.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago pa.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Kailangang i-transpile ang mga abstract circuit dahil limitado lang ang set ng basis gates ng mga QPU at hindi nito maaaring isagawa ang mga arbitrary na operasyon. Ang tungkulin ng transpiler ay baguhin ang mga arbitrary circuit para mapatakbo ang mga ito sa isang tinukoy na QPU. Ginagawa ito sa pamamagitan ng pagsasalin ng mga circuit sa mga sinusuportahang basis gates, at sa pag-introduce ng mga SWAP gate kung kinakailangan, para matugma ang connectivity ng circuit sa QPU.

Tulad ng ipinaliwanag sa [Transpile with pass managers](transpile-with-pass-managers), maaari kang gumawa ng [pass manager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) gamit ang function na [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) at magpasa ng circuit o listahan ng mga circuit sa [run](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) method nito para i-transpile ang mga ito. Maaari mong tawagan ang `generate_preset_pass_manager` na nagpapasa lamang ng optimization level at backend, na pinipiling gamitin ang mga default para sa lahat ng iba pang opsyon, o maaari kang magpasa ng karagdagang mga argumento sa function para pinong-ayusin ang transpilation.

## Pangunahing paggamit nang walang mga parameter
Sa halimbawang ito, nagpapasa tayo ng circuit at target na QPU sa transpiler nang hindi tinukoy ang anumang karagdagang parameter.

Gumawa ng circuit at tingnan ang resulta:

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import grover_operator, DiagonalGate
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

# Create circuit to test transpiler on
oracle = DiagonalGate([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(grover_operator(oracle))

# Add measurements to the circuit
qc.measure_all()

# View the circuit
qc.draw(output="mpl")

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg)

I-transpile ang circuit at tingnan ang resulta:

In [2]:
from qiskit.transpiler import generate_preset_pass_manager

# Specify the QPU to target
backend = FakeSherbrooke()

# Transpile the circuit
pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend
)
transpiled_circ = pass_manager.run(qc)

# View the transpiled circuit
transpiled_circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg)

## Lahat ng available na parameter
Narito ang lahat ng available na parameter para sa function na [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager). May dalawang klase ng mga argumento: ang mga naglalarawan sa target ng compilation, at ang mga nakaka-impluwensya sa paraan ng paggawa ng transpiler.

Lahat ng parameter maliban sa `optimization_level` ay opsyonal. Para sa buong detalye, tingnan ang [Transpiler API documentation](https://docs.quantum.ibm.com/api/qiskit/transpiler#transpiler-api).

- `optimization_level` (int) - Gaano karaming optimization ang isasagawa sa mga circuit. Integer sa hanay (0 - 3). Mas mataas na mga antas ang gumagawa ng mas optimized na mga circuit, sa gastos ng mas matagal na transpilation time. Tingnan ang [Set transpiler optimization level](set-optimization) para sa mas maraming detalye.

### Mga parameter na ginagamit para ilarawan ang compilation target:
Ang mga argumento na ito ay naglalarawan sa target na QPU para sa circuit execution, kasama ang impormasyon tulad ng coupling map ng QPU (na naglalarawan ng connectivity ng mga qubit), ang mga basis gate na sinusuportahan ng QPU, at ang mga error rate ng mga gate.

Marami sa mga parameter na ito ay inilarawan nang detalyado sa [Commonly used parameters for transpilation](common-parameters).

<details>
  <summary>
    **Mga parameter ng QPU (`Backend`)**
  </summary>

**Mga parameter ng Backend** - Kung tinukoy mo ang `backend`, hindi mo na kailangang tukuyin ang `target` o anumang iba pang backend options. Gayundin, kung tinukoy mo ang `target`, hindi mo na kailangang tukuyin ang `backend` o anumang iba pang backend options.
- `backend` (Backend) - Kung ito ay nakatakda, kino-compile ng transpiler ang input circuit sa device na ito. Kung may anumang iba pang opsyon na nakatakda na nakaka-apekto sa mga setting na ito, tulad ng `coupling_map`, ino-override nito ang mga setting mula sa `backend`.
- `target` (Target) - Isang backend transpiler target. Karaniwang tinukoy ito bilang bahagi ng backend argument, ngunit kung mano-manong nagawa mo ang isang Target object, maaari mo itong tukuyin dito. Ino-override nito ang target mula sa `backend`.
- `backend_properties` (BackendProperties) - Mga katangian na ibinalik ng isang QPU, kasama ang impormasyon tungkol sa mga gate error, readout error, qubit coherence time, at iba pa. Hanapin ang QPU na nagbibigay ng impormasyong ito sa pamamagitan ng pagpapatakbo ng `backend.properties()`.
- `timing_constraints` (Dict[str, int] | None) - Isang opsyonal na hardware restriction sa instruction time resolution. Ang impormasyong ito ay ibinibigay ng QPU configuration. Kung walang restriction ang QPU sa instruction time allocation, ang `timing_constraints` ay `None` at walang adjustment na isinasagawa. Maaaring mag-ulat ang QPU ng set ng mga restriction, katulad ng:
    - `granularity`: Isang integer value na kumakatawan sa minimum na pulse gate resolution sa mga unit ng dt. Ang isang user-defined pulse gate ay dapat may duration na multiple ng granularity value na ito.
    - `min_length`: Isang integer value na kumakatawan sa minimum na pulse gate length sa mga unit ng dt. Ang isang user-defined pulse gate ay dapat na mas mahaba kaysa sa length na ito.
    - `pulse_alignment`: Isang integer value na kumakatawan sa time resolution ng starting time ng gate instruction. Ang mga gate instruction ay dapat magsimula sa oras na multiple ng value na ito.
    - `acquire_alignment`: Isang integer value na kumakatawan sa time resolution ng starting time ng measure instruction. Ang measure instruction ay dapat magsimula sa oras na multiple ng value na ito.
</details>

<details>
  <summary>
    **Mga parameter ng Layout at topology**
  </summary>

- `basis_gates` (List[str] | None) - Listahan ng mga basis gate name na i-unroll. Halimbawa ['u1', 'u2', 'u3', 'cx']. Kung `None`, huwag mag-unroll.
- `coupling_map` (CouplingMap | List[List[int]]) - Directed coupling map (maaaring custom) para i-target sa mapping. Kung symmetric ang coupling map, kailangan tukuyin ang dalawang direksyon. Sinusuportahan ang mga format na ito:
    - CouplingMap instance
    - List - dapat ibigay bilang adjacency matrix, kung saan tinukoy ng bawat entry ang lahat ng directed two-qubit interaction na sinusuportahan ng QPU. Halimbawa: [[0, 1], [0, 3], [1, 2], [1, 5], [2, 5], [4, 1], [5, 3]]
- `inst_map` (List[InstructionScheduleMap] | None) - Mapping ng mga circuit operation sa pulse schedule. Kung `None`, ginagamit ang `instruction_schedule_map` ng QPU.
</details>

### Mga parameter na ginagamit para maimpluwensyahan kung paano gumagana ang transpiler
Ang mga parameter na ito ay nakaka-apekto sa mga tiyak na yugto ng transpilation. Ang ilan sa mga ito ay maaaring makaka-apekto sa maraming yugto, ngunit inilista lamang sa ilalim ng isang yugto para sa pagiging simple. Kung tumukoy ka ng argumento, tulad ng `initial_layout` para sa mga qubit na gusto mong gamitin, ang value na iyon ay ino-override ang lahat ng pass na maaaring baguhin ito. Sa ibang salita, hindi babaguhin ng transpiler ang anumang mano-manong tinukoy mo. Para sa mga detalye tungkol sa mga tiyak na yugto, tingnan ang [Transpiler stages](transpiler-stages).

<details>
  <summary>
    **Yugto ng Initialization**
  </summary>

- `hls_config` (HLSConfig) - Isang opsyonal na configuration class na `HLSConfig` na direktang ipinasa sa `HighLevelSynthesis` transformation pass. Ang configuration class na ito ay nagbibigay-daan sa iyo na tukuyin ang mga listahan ng synthesis algorithm at ang kanilang mga parameter para sa iba't ibang high-level na object.
- `init_method` (str) - Ang plugin name na gagamitin para sa initialization stage. By default, hindi ginagamit ang external plugin. Maaari kang makakita ng listahan ng mga naka-install na plugin sa pamamagitan ng pagpapatakbo ng `list_stage_plugins()` na may `init` para sa stage name argument.
- `unitary_synthesis_method` (str) - Ang pangalan ng unitary synthesis method na gagamitin. By default, ginagamit ang `default`. Maaari kang makakita ng listahan ng mga naka-install na plugin sa pamamagitan ng pagpapatakbo ng `unitary_synthesis_plugin_names()`.
- `unitary_synthesis_plugin_config` (dict) - Isang opsyonal na configuration dictionary na direktang ipinasa sa unitary synthesis plugin. By default, walang epekto ang setting na ito dahil ang default na unitary synthesis method ay hindi tumatanggap ng custom configuration. Ang pag-apply ng custom configuration ay kailangan lamang kapag ang isang unitary synthesis plugin ay tinukoy gamit ang `unitary_synthesis` argument. Dahil ito ay custom para sa bawat unitary synthesis plugin, sumangguni sa dokumentasyon ng plugin para malaman kung paano gamitin ang opsyong ito.
</details>

<details>
  <summary>
    **Yugto ng Layout**
  </summary>

- `initial_layout` (Layout | Dict | List) - Unang posisyon ng mga virtual qubit sa mga physical qubit. Kung ang layout na ito ay gagawing compatible ang circuit sa mga constraint ng `coupling_map`, gagamitin ito. Hindi ginagarantiyahan na magiging pareho ang final layout, dahil maaaring mag-permute ng mga qubit ang transpiler sa pamamagitan ng mga swap o iba pang paraan. Para sa buong detalye, tingnan ang [Initial layout section.](common-parameters#initial-layout)
- `layout_method` (str) - Pangalan ng layout selection pass (`default`, `dense`, `sabre`, at `trivial`). Maaari rin itong maging external plugin name para gamitin sa layout stage. Maaari kang makakita ng listahan ng mga naka-install na plugin sa pamamagitan ng pagpapatakbo ng `list_stage_plugins()` na may `layout` para sa `stage_name` argument. Ang default value ay `sabre`.
</details>

<details>
  <summary>
    **Yugto ng Routing**
  </summary>

- `routing_method` (str) - Pangalan ng routing pass (`basic`, `lookahead`, `default`, `sabre`, o `none`). Maaari rin itong maging external plugin name para gamitin sa routing stage. Maaari kang makakita ng listahan ng mga naka-install na plugin sa pamamagitan ng pagpapatakbo ng `list_stage_plugins()` na may `routing` para sa `stage_name` argument. Ang default value ay `sabre`.
</details>

<details>
  <summary>
    **Yugto ng Translation**
  </summary>

- `translation_method` (str) - Pangalan ng translation pass (`default`, `synthesis`, `translator`, `ibm_backend`, `ibm_dynamic_circuits`, `ibm_fractional`). Maaari rin itong maging external plugin name para gamitin sa translation stage. Maaari kang makakita ng listahan ng mga naka-install na plugin sa pamamagitan ng pagpapatakbo ng `list_stage_plugins()` na may `translation` para sa `stage_name` argument. Ang default value ay `translator`.
</details>

<details>
  <summary>
    **Yugto ng Optimization**
  </summary>

- `approximation_degree` (float, sa hanay 0-1 | None) - Heuristic dial na ginagamit para sa circuit approximation (1.0 = walang approximation, 0.0 = maximum na approximation). Ang default value ay 1.0. Ang pagtukoy ng `None` ay nagtatakda ng approximation degree sa iniulat na error rate. Tingnan ang [Approximation degree section](common-parameters#approx-degree) para sa mas maraming detalye.
- `optimization_method` (str) - Ang plugin name na gagamitin para sa optimization stage. By default, hindi ginagamit ang external plugin. Maaari kang makakita ng listahan ng mga naka-install na plugin sa pamamagitan ng pagpapatakbo ng `list_stage_plugins()` na may `optimization` para sa `stage_name` argument.
</details>

<details>
  <summary>
    **Yugto ng Scheduling**
  </summary>

- `scheduling_method` (str) - Pangalan ng scheduling pass. Maaari rin itong maging external plugin name para gamitin sa scheduling stage. Maaari kang makakita ng listahan ng mga naka-install na plugin sa pamamagitan ng pagpapatakbo ng `list_stage_plugins()` na may `scheduling` para sa `stage_name` argument.
  * 'as_soon_as_possible': I-schedule ang mga instruction nang masigasig, nang maaga hangga't maaari sa isang qubit resource (alias: `asap`).
  * 'as_late_as_possible': I-schedule ang mga instruction nang huli, iyon ay, pinapanatiling nasa ground state ang mga qubit hangga't maaari (alias: `alap`). Ito ang default.
</details>

<details>
  <summary>
    **Pagpapatupad ng Transpiler**
  </summary>

- `seed_transpiler` (int) - Nagtatakda ng random seed para sa mga stochastic na bahagi ng transpiler.
</details>

Ang mga sumusunod na default value ay ginagamit kung hindi mo tinukoy ang alinman sa mga parameter sa itaas. Sumangguni sa [API reference page](../api/qiskit/transpiler_preset) ng method para sa karagdagang impormasyon:

In [3]:
generate_preset_pass_manager(
    optimization_level=1,
    backend=None,
    target=None,
    basis_gates=None,
    coupling_map=None,
    initial_layout=None,
    layout_method=None,
    routing_method=None,
    translation_method=None,
    scheduling_method=None,
    approximation_degree=1.0,
    seed_transpiler=None,
    unitary_synthesis_method="default",
    unitary_synthesis_plugin_config=None,
    hls_config=None,
    init_method=None,
    optimization_method=None,
)

## Next steps

<Admonition type="tip" title="Recommendations">
    - Learn how to [Set the optimization level](set-optimization).
    - Review more [Commonly used parameters](common-parameters).
    - Learn how to [Set the optimization level when using Qiskit Runtime.](./runtime-options-overview)
    - Visit the [Transpile with pass managers](transpile-with-pass-managers) topic.
    - For examples, see [Representing quantum computers.](./represent-quantum-computers)
    - Learn [how to transpile circuits](/docs/guides/circuit-transpilation-settings) as part of the Qiskit patterns workflow using Qiskit Runtime.
    - Review the [Transpile API documentation](/docs/api/qiskit/transpiler).
</Admonition>